# Train the Philly Water Watch garbage detector (Google Colab)

1. **Runtime → Change runtime type → T4 GPU**, then Save.
2. Upload `WaterDataset.zip` to the top level of your Google Drive (**My Drive**).
3. **Runtime → Run all.** Training takes about 30–60 minutes.
4. The last cell downloads `garbage.pt`. Put it in `backend/models/garbage.pt` and restart the API.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
%pip install -q ultralytics

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ZIP = '/content/drive/MyDrive/WaterDataset.zip'   # change if you put it elsewhere

In [ ]:
# Unzip to fast local disk and move YOLO labels where Ultralytics expects them:
#   fml/images/<split>/x.jpg  ->  fml/labels/<split>/x.txt
import os, shutil, zipfile, pathlib
root = pathlib.Path('/content/fml')
if not root.exists():
    zipfile.ZipFile(ZIP).extractall('/content')
for split in ('train', 'val', 'test'):
    src = root / 'labels' / 'yolo_format' / split
    dst = root / 'labels' / split
    if src.exists() and not dst.exists():
        shutil.move(str(src), str(dst))
    print(split, len(list((root / 'images' / split).glob('*.jpg'))), 'images,',
          len(list(dst.glob('*.txt'))), 'labels')

pathlib.Path('/content/data.yaml').write_text(f'''path: {root}
train: images/train
val: images/val
test: images/test
names:
  0: garbage
''')

In [ ]:
from ultralytics import YOLO
# Photos are 1920x1080 with many small items, so train at 960px.
# Lower imgsz to 640 (or epochs to 30) if you run short on Colab time.
model = YOLO('yolo11n.pt')
model.train(data='/content/data.yaml', epochs=60, imgsz=960, batch=16, patience=15,
            project='/content/runs', name='garbage', exist_ok=True)

In [ ]:
# Held-out test set score (mAP50 is the headline number)
best = YOLO('/content/runs/garbage/weights/best.pt')
metrics = best.val(data='/content/data.yaml', split='test', imgsz=960)
print(f'test mAP50={metrics.box.map50:.3f}  mAP50-95={metrics.box.map:.3f}  '
      f'precision={metrics.box.mp:.3f}  recall={metrics.box.mr:.3f}')

In [ ]:
# Save a copy to Drive and download it
import shutil
from google.colab import files
shutil.copy('/content/runs/garbage/weights/best.pt', '/content/garbage.pt')
shutil.copy('/content/garbage.pt', '/content/drive/MyDrive/garbage.pt')
files.download('/content/garbage.pt')